1 importy </br>
2 wczytanie splitów</br>
3 filtr young / old</br>
4 tf.data dataset</br>
5 generator</br>
6 discriminator</br>
7 loss functions</br>
8 training loop</br>

In [1]:
import tensorflow as tf

In [2]:

print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
print(tf.config.list_physical_devices('GPU'))

Num GPUs Available: 1
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
class InstanceNormalization(tf.keras.layers.Layer):
    def __init__(self, epsilon=1e-5):
        super().__init__()
        self.epsilon = epsilon

    def build(self, input_shape):
        self.gamma = self.add_weight(
            shape=(input_shape[-1],),
            initializer="ones",
            trainable=True,
        )
        self.beta = self.add_weight(
            shape=(input_shape[-1],),
            initializer="zeros",
            trainable=True,
        )

    def call(self, x):
        mean, variance = tf.nn.moments(x, axes=[1, 2], keepdims=True)
        x = (x - mean) / tf.sqrt(variance + self.epsilon)
        return self.gamma * x + self.beta

In [4]:
import pandas as pd

train_df = pd.read_csv("../data/metadata/train.csv")
val_df = pd.read_csv("../data/metadata/val.csv")
test_df = pd.read_csv("../data/metadata/test.csv")

FileNotFoundError: [Errno 2] No such file or directory: '../data/metadata/train.csv'

In [ ]:
import tensorflow as tf
from tensorflow.keras import mixed_precision


IMG_SIZE = 128
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

mixed_precision.set_global_policy("mixed_float16")

print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


Load the splits

In [ ]:
#load the splits
young_df = train_df[train_df["age_group"] == "young"]
old_df = train_df[train_df["age_group"] == "old"]

Image Loader

In [ ]:
def load_image(path):

    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)

    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))

    img = (tf.cast(img, tf.float32) / 127.5) - 1
            #for gan we use [-1, 1] range

    return img

Dataset

In [ ]:
young_dataset = (
    tf.data.Dataset.from_tensor_slices(young_df["image_path"])
    .shuffle(1000)
    .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(1)
    .cache()
    .prefetch(tf.data.AUTOTUNE)
).take(50)

old_dataset = (
    tf.data.Dataset.from_tensor_slices(old_df["image_path"])
    .shuffle(1000)
    .map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(1)
    .cache()
    .prefetch(tf.data.AUTOTUNE)
).take(50)

Archtecture is:</br>
</br>
G : young → old </br>
F : old → young</br>
</br>
D_Y : discriminator young</br>
D_O : discriminator old</br>

Downsample

In [ ]:

from tensorflow.keras import layers

def downsample(filters, size, apply_instancenorm=True):

    initializer = tf.random_normal_initializer(0., 0.02)

    result = tf.keras.Sequential()
    result.add(
        layers.Conv2D(filters, size, strides=2, padding="same",
                      kernel_initializer=initializer, use_bias=False)
    )

    if apply_instancenorm:
        result.add(InstanceNormalization())

    result.add(layers.LeakyReLU())

    return result

Upsample

In [ ]:
def upsample(filters, size):

    initializer = tf.random_normal_initializer(0., 0.02)

    result = tf.keras.Sequential()
    result.add(
        layers.Conv2DTranspose(filters, size, strides=2,
                               padding="same",
                               kernel_initializer=initializer,
                               use_bias=False)
    )

    result.add(InstanceNormalization())
    result.add(layers.ReLU())

    return result

Instanse Normalization

In [ ]:
class InstanceNormalization(tf.keras.layers.Layer):
    def __init__(self, epsilon=1e-5):
        super().__init__()
        self.epsilon = epsilon

    def build(self, input_shape):
        self.gamma = self.add_weight(
            shape=(input_shape[-1],),
            initializer="ones",
            trainable=True,
        )
        self.beta = self.add_weight(
            shape=(input_shape[-1],),
            initializer="zeros",
            trainable=True,
        )

    def call(self, x):
        mean, variance = tf.nn.moments(x, axes=[1, 2], keepdims=True)
        x = (x - mean) / tf.sqrt(variance + self.epsilon)
        return self.gamma * x + self.beta

Reflection Padding


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

class ReflectionPadding2D(layers.Layer):
    def __init__(self, padding=(1,1), **kwargs):
        self.padding = tuple(padding)
        super().__init__(**kwargs)

    def call(self, x):
        padding_width, padding_height = self.padding
        return tf.pad(
            x,
            [[0,0],
             [padding_height,padding_height],
             [padding_width,padding_width],
             [0,0]],
            mode="REFLECT"
        )

ResNet Block

In [ ]:
def resnet_block(x, filters, use_dropout=True):

    skip = x

    y = ReflectionPadding2D((1,1))(x)
    y = layers.Conv2D(filters, kernel_size=3)(y)
    y = InstanceNormalization()(y)
    y = layers.ReLU()(y)

    if use_dropout:
        y = layers.Dropout(0.5)(y)

    y = ReflectionPadding2D((1,1))(y)
    y = layers.Conv2D(filters, kernel_size=3)(y)
    y = InstanceNormalization()(y)

    out = layers.Add()([skip, y])

    return out

Generator model

In [ ]:


def build_generator():

    inputs = layers.Input(shape=[IMG_SIZE,IMG_SIZE,3])

    # initial conv
    x = layers.Conv2D(64, 7, padding="same")(inputs)
    x = InstanceNormalization()(x)
    x = layers.ReLU()(x)

    # downsample
    x = layers.Conv2D(128, 3, strides=2, padding="same")(x)
    x = InstanceNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(256, 3, strides=2, padding="same")(x)
    x = InstanceNormalization()(x)
    x = layers.ReLU()(x)

    # RESNET BLOCKS
    for _ in range(6):
        x = resnet_block(x, 256)

    # upsample
    x = layers.Conv2DTranspose(128, 3, strides=2, padding="same")(x)
    x = InstanceNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2DTranspose(64, 3, strides=2, padding="same")(x)
    x = InstanceNormalization()(x)
    x = layers.ReLU()(x)

   # Output layer
    x = ReflectionPadding2D((3,3))(x)
    outputs = layers.Conv2D(
        3,
        kernel_size=7,
        activation="tanh",
        dtype="float32"

    )(x)

    return tf.keras.Model(inputs, outputs)

Create 2 generators

G : young → old </br>
F : old → young

In [ ]:
generator_g = build_generator()
generator_f = build_generator()

Discriminator model

In [ ]:
def build_discriminator():

    initializer = tf.random_normal_initializer(0.,0.02)

    inp = layers.Input(shape=[IMG_SIZE,IMG_SIZE,3])

    x = downsample(64,4,False)(inp)
    x = downsample(128,4)(x)
    x = downsample(256,4)(x)

    x = layers.Conv2D(512,4,strides=1,padding="same",
                      kernel_initializer=initializer)(x)

    x = layers.LeakyReLU()(x)

    x = layers.Conv2D(1,4,strides=1,padding="same",
                      kernel_initializer=initializer)(x)

    return tf.keras.Model(inp,x)

Create 2 discriminators

In [ ]:
discriminator_x = build_discriminator()
discriminator_y = build_discriminator()

## Loss functions

binery creossentropy


In [ ]:
loss_obj = tf.keras.losses.BinaryCrossentropy(from_logits=True)

In [ ]:
def discriminator_loss(real, generated):

    real_loss = loss_obj(tf.ones_like(real), real)
    generated_loss = loss_obj(tf.zeros_like(generated), generated)

    total_disc_loss = real_loss + generated_loss

    return total_disc_loss * 0.5

Generator adversial loss

In [ ]:
def generator_loss(generated):

    return loss_obj(tf.ones_like(generated), generated)

Cycle consistency loss

In [ ]:
LAMBDA_CYCLE = 10
LAMBDA_IDENTITY = 5

In [ ]:
def cycle_loss(real_image, cycled_image):

    loss = tf.reduce_mean(tf.abs(real_image - cycled_image))

    return LAMBDA_CYCLE * loss

Identity loss

In [ ]:
def identity_loss(real_image, same_image):

    loss = tf.reduce_mean(tf.abs(real_image - same_image))

    return LAMBDA_IDENTITY * loss

Optimizers

In [ ]:
generator_g_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
generator_f_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

discriminator_x_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
discriminator_y_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

Image pool (Fake image buffer)

In [ ]:
import random

class ImagePool:

    def __init__(self, pool_size=50):

        self.pool_size = pool_size
        self.images = []

    def query(self, image):

        if len(self.images) < self.pool_size:
            self.images.append(image)
            return image

        if random.random() > 0.5:

            idx = random.randint(0, self.pool_size - 1)
            tmp = self.images[idx]
            self.images[idx] = image

            return tmp

        else:
            return image

In [ ]:
fake_x_pool = ImagePool()
fake_y_pool = ImagePool()

Train step

In [ ]:
@tf.function
def train_step(real_x, real_y):

    with tf.GradientTape(persistent=True) as tape:

        fake_y = generator_g(real_x, training=True)
        cycled_x = generator_f(fake_y, training=True)

        fake_x = generator_f(real_y, training=True)
        cycled_y = generator_g(fake_x, training=True)

        same_x = generator_f(real_x, training=True)
        same_y = generator_g(real_y, training=True)

        disc_real_x = discriminator_x(real_x, training=True)
        disc_real_y = discriminator_y(real_y, training=True)


        fake_x = fake_x_pool.query(fake_x)
        fake_y = fake_y_pool.query(fake_y)

        disc_fake_x = discriminator_x(fake_x)
        disc_fake_y = discriminator_y(fake_y)

        gen_g_loss = generator_loss(disc_fake_y)
        gen_f_loss = generator_loss(disc_fake_x)

        total_cycle_loss = cycle_loss(real_x, cycled_x) + cycle_loss(real_y, cycled_y)

        total_gen_g_loss = gen_g_loss + total_cycle_loss + identity_loss(real_y, same_y)
        total_gen_f_loss = gen_f_loss + total_cycle_loss + identity_loss(real_x, same_x)

        disc_x_loss = discriminator_loss(disc_real_x, disc_fake_x)
        disc_y_loss = discriminator_loss(disc_real_y, disc_fake_y)

    generator_g_gradients = tape.gradient(
        total_gen_g_loss,
        generator_g.trainable_variables
    )

    generator_f_gradients = tape.gradient(
        total_gen_f_loss,
        generator_f.trainable_variables
    )

    discriminator_x_gradients = tape.gradient(
        disc_x_loss,
        discriminator_x.trainable_variables
    )

    discriminator_y_gradients = tape.gradient(
        disc_y_loss,
        discriminator_y.trainable_variables
    )

    generator_g_optimizer.apply_gradients(
        zip(generator_g_gradients, generator_g.trainable_variables)
    )

    generator_f_optimizer.apply_gradients(
        zip(generator_f_gradients, generator_f.trainable_variables)
    )

    discriminator_x_optimizer.apply_gradients(
        zip(discriminator_x_gradients, discriminator_x.trainable_variables)
    )

    discriminator_y_optimizer.apply_gradients(
        zip(discriminator_y_gradients, discriminator_y.trainable_variables)
    )

Generate previews

In [ ]:
#generate previews

import matplotlib.pyplot as plt

def generate_images(model, test_input):

    prediction = model(test_input, training=False)

    plt.figure(figsize=(6,3))

    display_list = [test_input[0], prediction[0]]
    title = ['Input', 'Generated']

    for i in range(2):
        plt.subplot(1,2,i+1)
        plt.title(title[i])

        img = (display_list[i] * 0.5) + 0.5
        plt.imshow(img)

        plt.axis('off')

    plt.show()

Checkpoints

In [ ]:
checkpoint = tf.train.Checkpoint(
    generator_g=generator_g,
    generator_f=generator_f,
    discriminator_x=discriminator_x,
    discriminator_y=discriminator_y,
    generator_g_optimizer=generator_g_optimizer,
    generator_f_optimizer=generator_f_optimizer,
    discriminator_x_optimizer=discriminator_x_optimizer,
    discriminator_y_optimizer=discriminator_y_optimizer
)

In [ ]:
checkpoint_manager = tf.train.CheckpointManager(
    checkpoint,
    './checkpoints',
    max_to_keep=5
)

In [ ]:
import shutil
import os

# Clear old checkpoints to avoid incompatible model loading
if os.path.exists('./checkpoints'):
    shutil.rmtree('./checkpoints')
    print("Cleared old checkpoints directory")


In [ ]:
sample = next(iter(young_dataset))
print(sample.shape)

(1, 128, 128, 3)


Training loop

In [ ]:
EPOCHS = 1

#for the previews
sample_young = next(iter(young_dataset))

for epoch in range(EPOCHS):

    for image_x, image_y in tf.data.Dataset.zip((young_dataset, old_dataset)):
        train_step(image_x, image_y)

    print("Epoch completed:", epoch)

    generate_images(generator_g, sample_young)
    checkpoint_manager.save()